We'll set everything up here before writing to a .py

In [3]:
# importing our base model
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torchvision

/Users/achrafbayi/Desktop/JARVIS-VLA-MINECRAFT/.venv.nosync/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1115 18:01:16.780000 26887 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [4]:
# I suspect this cell to somehow crash everytime
base_model = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-2B-Instruct", dtype=torch.bfloat16, device_map="mps")
base_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct") 
tokenizer = base_processor.tokenizer

Loading weights: 100%|██████████| 625/625 [00:00<00:00, 841.36it/s]


In [6]:
# Ok let's now start working on Phase I which uses the : https://huggingface.co/datasets/CraftJarvis/minecraft-vlp dataset
from huggingface_hub import snapshot_download, hf_hub_download
import huggingface_hub

# here for phase I we just need a single file from the repo
# macos has been fucking around a lot with the download of our files
import os
path = "./data_phase_one.nosync/mc-qa-250312.jsonl"
DATALESS = 0x40000000  # macOS flag for iCloud placeholder files

if not os.path.exists(path) or os.stat(path).st_flags & DATALESS:
    path = hf_hub_download(repo_id="CraftJarvis/minecraft-vlp", filename="mc-qa-250312.jsonl",
                           repo_type="dataset", local_dir="./data_phase_one.nosync")
data_step_one = path
# we want to train to predict : P(assistant | system, user)

In [4]:
# for reference here is what one line of that json looks like :

'''
{"id": "qa-f89343ef-d687-49b5-95d5-1d0e9d63da18", "label": ["qa", "wiki", "self-instruct"], "image": [], 
"conversations": 
[{"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant that can answer questions and help with tasks in Minecraft.\n"}]}, 
{"role": "user", "content": [{"type": "text", "text": "How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?"}]}, 
{"role": "assistant", "content": [{"type": "text", "text": "You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune."}]}]}


No image context for phase 1 ofc
'''

#Let's real quickly try to parse that line fine before doing it on the whole big ass json file
import json 

with open(data_step_one) as f:
    lines = f.readlines()

line = lines[12]
#print(line)

sample = json.loads(line)
sample["conversations"][1]["content"][0]['text'] # How do you enchant items in Minecraft, and what are some of the enchantments that can be applied? 
# the question
system=sample["conversations"][0]["content"][0]['text']
user = sample["conversations"][1]["content"][0]['text']
assistant = sample["conversations"][2]["content"][0]['text']

# format we want (with the Qwen tokens/markers and the format it was pretrained on)

pre_tokenization_datapoint = '<|im_start|>system\n'+system+'<|im_end|>\n'+'<|im_start|>user\n'+user+'<|im_end|>\n'+'<|im_start|>assistant\n'+assistant+'<|im_end|>\n'
print(pre_tokenization_datapoint)

'''
Good :

<|im_start|>system
You are a helpful assistant that can answer questions and help with tasks in Minecraft.
<|im_end|>
<|im_start|>user
How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>
<|im_start|>assistant
You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>
'''



<|im_start|>system
You are a helpful assistant that can answer questions and help with tasks in Minecraft.
<|im_end|>
<|im_start|>user
How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>
<|im_start|>assistant
You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>



'\nGood :\n\n<|im_start|>system\nYou are a helpful assistant that can answer questions and help with tasks in Minecraft.\n<|im_end|>\n<|im_start|>user\nHow do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>\n<|im_start|>assistant\nYou can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>\n'

In [5]:
# now that we got the format right, let's do it over the whole json, and store it

with open('./data_phase_one.nosync/phase_one_formatted.jsonl', 'w') as f: #w
    for line in lines:
        dct = json.loads(line)
        system=dct["conversations"][0]["content"][0]['text']
        user = dct["conversations"][1]["content"][0]['text']
        assistant = dct["conversations"][2]["content"][0]['text']
        text = '<|im_start|>system\n'+system+'<|im_end|>\n'+'<|im_start|>user\n'+user+'<|im_end|>\n'+'<|im_start|>assistant\n'+assistant+'<|im_end|>\n'
        f.write(json.dumps({'datapoint':text})+'\n')


In [6]:
# now let's set up our proper dataset
# we want to keep the sentences separate, we hence want sumn like an array of arrays of tokens
import json 

with open('./data_phase_one.nosync/phase_one_formatted.jsonl', 'r') as f:
    buckets = [json.loads(line)['datapoint'] for line in f.readlines()]

#print(buckets[2]) we get what we want

token_buckets = [tokenizer(point, add_special_tokens=False) for point in buckets]
print(len(token_buckets))

print(token_buckets[3]) # good, we get the token ids

275905
{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 429, 646, 4226, 4755, 323, 1492, 448, 9079, 304, 40230, 624, 151645, 198, 151644, 872, 198, 4340, 653, 498, 1855, 264, 3731, 10812, 16224, 304, 40230, 323, 1128, 525, 1045, 315, 279, 2155, 5711, 369, 432, 30, 151645, 198, 151644, 77091, 198, 641, 40230, 11, 3731, 10812, 374, 264, 4911, 5101, 429, 646, 387, 1483, 311, 1855, 6351, 45021, 323, 7611, 13, 3731, 10812, 374, 1483, 311, 29282, 2355, 11, 20276, 23783, 11, 323, 68611, 5257, 9079, 304, 279, 1809, 13, 5692, 594, 264, 6770, 8474, 389, 1246, 311, 1855, 264, 3731, 10812, 16224, 323, 1045, 2155, 5711, 369, 432, 1447, 334, 16, 13, 48995, 3731, 10812, 25, 1019, 256, 481, 2014, 1191, 11, 498, 1184, 311, 10485, 3731, 10812, 16144, 10010, 11, 892, 525, 5990, 1730, 5538, 25260, 13, 1446, 3278, 1184, 458, 11001, 3735, 49739, 476, 2664, 311, 6530, 3731, 10812, 15797, 382, 334, 17, 13, 4230, 3731, 10812, 31993, 25, 1019, 256, 481, 9646, 498, 614, 3731, 10812, 16144, 11, 104

In [7]:
# ok, now that we have that, we want to build our dataset for GPT-like training, where
# for each x, y=x+1 (in token indexing)
# but here we also don't want to train the llm to predict the user and system tokens
# hence we need our shortest substring to be the following : <System content and tokens><User content and tokens><|im_start|>assistant\n
# and build on from that to the last token
# the string that delimits (and that is unique in each datapoint which we can use as the "border") in every datapoint is the following
# <|im_start|>assistant\n
# let's tokenize it to make it the magic tokens that signal the start of or sequence
start_tokens = tokenizer("<|im_start|>assistant\n", add_special_tokens=False)['input_ids']
print(start_tokens)
# [151644, 77091, 198] those 3 are the start tokens, hence we'd want to look for those 3 tokens before the start of each of our sequences

magic_tokens = [151644, 77091, 198]
dsx_idx = []
for datapoint, n in zip(token_buckets, range(len(token_buckets))):
        # we'll use sliding window on each sequence to detect starting index
        datapoint = datapoint['input_ids']
        x = -1
        length = len(datapoint)
        for i in range(length-2):
            if datapoint[i] == 151644 and datapoint[i+1] == 77091: # our magic tokens
                if datapoint[i+2]==198: # i.e the token for \n, meaning it didn't get merged into a bigger token, we skip it
                    x = i+3 # the convention we'll use here is that x stores the ds[:x] for x in range(x, len-1)
                    # and here the convention will be that y = ds[x] simply as the next token we want to preditct
                else:
                    x = i+2
        if x==-1 : 
            print(x)
            print(n) # to know which line is dirty 
            print(buckets[n])
            print(token_buckets[n])
            raise RuntimeError("we have empty x's and y's")
        else: dsx_idx.append(x)
'''
Btw here, the reason why we did not include the 'and datapoint[i+2] == 198' in our conditional is that we encountered cases where that \ n char 
got merged into another 'bigger/newer' token (in the BPE sense) and hence was missing and threw the error, hence that (squence at index 17067)

if x==[] or y==[]: 

is a really good guard as it allows us to stall whenever something like that happens (initially we had the and datapoint[i+2] == 198 component in our
sliding window boolean)

'''

# checking if 17067 is now fixed :

print(tokenizer.decode(token_buckets[17067]['input_ids'][dsx_idx[17067]+1:], add_special_tokens=False)) # do indeed get the right answer which is reassuring


[151644, 77091, 198]


<>:35: SyntaxWarning: invalid escape sequence '\ '
<>:35: SyntaxWarning: invalid escape sequence '\ '
/var/folders/13/1fmmyq5j5n303885dkdjlzn00000gn/T/ipykernel_8926/1560089060.py:35: SyntaxWarning: invalid escape sequence '\ '
  Btw here, the reason why we did not include the 'and datapoint[i+2] == 198' in our conditional is that we encountered cases where that \ n char


Minecraft offers endless possibilities for creative building ideas that utilize redstone. Redstone is a versatile resource that can be used to create complex contraptions, automation systems, and interactive structures. Here are some creative building ideas to get you started:

1. **Hidden Entrance or Base:** Create a hidden entrance to your base using redstone mechanisms like piston doors, secret levers, or pressure plates. Combine them with redstone lamps or note blocks for added effects.

2. **Automated Farm:** Build an automated farm that uses redstone to harvest and replant crops, such as wheat, carrots, or potatoes. You can also create a fully automatic tree farm or animal breeding system.

3. **Elevator or Escalator:** Design a redstone-powered elevator or escalator that allows you to move between different floors of your building effortlessly. Elevators can be especially impressive when designed with multiple floors and destinations.

4. **Adventure Map or Puzzle:** Craft an ad

In [8]:
# we basically now have with token buckets and dsx_idx the right dataset and the indx on which to start training
import numpy as np
ds = list(zip(dsx_idx, (tok['input_ids'] for tok in token_buckets)))

import random
random.Random(67).shuffle(ds) #shuffling for sgd in case the user query -> responses were in some kind of order, and to split cleanly

breaker = int(len(ds)*0.99) # we only keep 1 percent for test set this time

train_ds = ds[:breaker]
test_ds = ds [breaker:]
print(len(test_ds))
print(len(max(ds, key=lambda x :len(x[1]))[1])) # below the paper's 3584 for phase I
# below the paper's 3584 for phase I
print(np.mean([len(x[1]) for x in ds])) # mean is 402
# so no need to ever cut a sequence

2760
1672
411.4454323045976


In [9]:
print(tokenizer.pad_token_id)
print(tokenizer.decode(tokenizer.pad_token_id)) # it is the EOF token ?

151643
<|endoftext|>


In [ ]:
# Phase I Training loop :

# first of all, we need to freeze all the ViT/vision related parameters of our Qwen VLM
import math

for p in base_model.model.visual.parameters():
    p.requires_grad = False


# then we set up the optimizer, learning rate and everything, we only want to pass in the parameters that have required_grad = True ofc
# as well as the parameters specified by the paper (I think default for AdamW B2 is like 0.999 so we do need to pass em explicitly)
trainable_params = [p for p in base_model.parameters() if p.requires_grad == True]
optimizer = torch.optim.AdamW(
    params = trainable_params,
    betas=(0.9, 0.95),
    eps=1e-8,
    lr=5e-6,
    weight_decay=0
)

# let's define our max_steps, since the paper doesn't specifiy the number of steps/epochs they used, I'll assume 1 epoch
print(len(train_ds))
B = 256
max_step = int(len(train_ds)/B) #273145

# now our cosine lr w/ warmup function we'll use in the loop
lr_max = 5e-6
warmup_steps = 200
# warmup of 200 steps
def get_lr(step):
    # here the paper didn't specify lr_min so for the function I'll just assume it to be 0
    if step <= warmup_steps:
        return step * (lr_max/200)
    else:
        return (1/2)*lr_max*(1+math.cos(((step-warmup_steps)/max_step)*math.pi))

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu'
import torch.nn.functional as F

# actually rather than building the batch inside the iters, we might want to make it into a function, maybe for later use
eval_every = 150

def build_batch_tensors(ds):
    max_seq_length = len(max(ds, key=lambda x : len(x[1]))[1]) -2 # -1 since last 2 tokens is never appart of any inputs
    # we want the longest sequence to be our shared seq length dimension across all sequence 3d tensors, and we'll add padding to all ones that don't meet that max
    # we want to get the pad token as I am not sure which one it is for qwen, we'll print it above, ok it is actually the EOF token so let's now build our batched tensor
    # and feed that bitch to QwenVL's llm
    x_batch_tensor = torch.empty(size=(len(ds), max_seq_length), dtype=torch.int64)
    y_batch_tensor = torch.empty(size=(len(ds), max_seq_length), dtype=torch.int64) # using int64 as our token ids are int64
    for tup, k in zip(ds, range(len(ds))):
        x_cat = torch.tensor(data=tup[1][:-2]) # (1,len) # skiping the end and \n last 2 characters
        pad = torch.full(size=(max_seq_length-len(x_cat),), fill_value=tokenizer.pad_token_id)
        x = torch.cat([x_cat, pad], dim=0)
        y_cat = torch.full(size=(tup[0]-1,), fill_value=-67) # out of range, negative
        actual_values = torch.tensor(data=[tup[1][w] for w in range(tup[0], len(tup[1])-1)])  # in range, the actual ground truth
        y_pad = torch.full(size=(max_seq_length - (len(tup[1])-2),), fill_value=-67) # padding with our future ignore index
        y = torch.cat([y_cat,actual_values,y_pad], dim=0)
        x_batch_tensor[k] = x
        y_batch_tensor[k] = y
    return x_batch_tensor,y_batch_tensor

# we'll also want to log our loss to draw nice curves
import os, time
run_id = time.strftime("%Y%m%d-%H%M%S")
log_dir = f"phase_one_logs.nosync/{run_id}"
os.makedirs(log_dir, exist_ok=True)

train_log = open(f"{log_dir}/train.csv", "w"); train_log.write("step,loss,lr\n")
test_log  = open(f"{log_dir}/test.csv", "w");  test_log.write("step,loss\n")

# we'll use this function to compute test loss
@torch.no_grad()
def test_loss(bs=16):
    base_model.eval()
    total, n_tok = 0.0, 0
    for i in range(0, len(test_ds), bs):
        x, y = build_batch_tensors(test_ds[i:i+bs])
        x, y = x.to(device), y.to(device)
        logits = base_model(input_ids=x).logits
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-67, reduction="sum").item()
        n_tok += (y != -67).sum().item()
    base_model.train()
    return total / n_tok


# as said in the paper we want 256-long batches (256 sequences)
# hence that will require 256 forward passes of the llm, and multiplex into 256*lenght of the sequences (post-system+user content) loss computations
base_model.train() # flipping the value
for step in range(max_step):
    # we need to set up the batch
    # let's remember that our ds is in the form [(idx, tokens)] where ids is where the assistant part of the sequence beings
    batch_ds = train_ds[(step)*B:(step+1)*B] # start is inclusive, end exclusive
    x, y = build_batch_tensors(batch_ds)

    # now that we have our batched tensor, let's push them through the llm of Qwen3VL

    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad() # zeroing out the gradient values first
    out = base_model(input_ids=x) # going from token_ids to the actual vectors happens in the model so we're gucci
    logits = out.logits # we get something of the shape (B, len, vocab) i.e one logit distribution per token which is what we wanted
    loss = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), # (B*len, vocab)
        y.reshape(-1), # (B*len)
        ignore_index=-67
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0) # gradient clipping
    for g in optimizer.param_groups: g["lr"] = get_lr(step)
    optimizer.step()

    train_log.write(f"{step},{loss.item()},{get_lr(step)}\n"); train_log.flush()
    if step % 50 == 0:
        print(f"step {step} | loss {loss.item():.4f} | lr {get_lr(step):.2e}")

    if step % eval_every == 0 or step == max_step - 1: # test loss every 200
        tl = test_loss()
        test_log.write(f"{step},{tl}\n"); test_log.flush()
        print(f"  test loss {tl:.4f}")

    if step == max_step-1: # since phase one is kind of short, it is ok to only save at the end
        base_model.save_pretrained(f"checkpoints/{run_id}/phase1")




273145


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG16GFamilyCommandBuffer: 0xdb6c04000>
    label = <none> 
    device = <AGXG16GDevice: 0x104435230>
        name = Apple M4 
    commandQueue = <AGXG16GFamilyCommandQueue: 0x99275c000>
        label = <none> 
        device = <AGXG16GDevice: 0x104435230>
            name = Apple M4 
    retainedReferences = 1


KeyboardInterrupt: 

In [10]:
# now starting with phase 2
from huggingface_hub import snapshot_download

# first off we need to unfreeze all the weights

for p in base_model.parameters():
    p.requires_grad = True
    

    # now we need to set up the visual tokens + instruction part, and we need to import it from the dataset

data_step_two = snapshot_download(
    repo_id="CraftJarvis/minecraft-vlp",
    repo_type="dataset",
    local_dir="./data_phase_two.nosync", 
    ignore_patterns=["mc-knowledge-valid.jsonl", "mc-qa-250312.jsonl", "hallucination.jsonl", ".gitattributes"]) # ignoring the phase 1 stuff


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 1057.97it/s]


In [11]:
# now that I have downloaded the data, we get to see the structure of it, we have 5 jsonl files with {"id": "6940125a-15fc-4776-9dc5-ae75cf0f4949", "label": ["vqa", "contractor", "Minecraft"], "image": ["images/mc-vqa-241102/6940125a-15fc-4776-9dc5-ae75cf0f4949.jpg"], "datetime": "24_11_02", "model": "gpt_4o", "conversations": [{"role": "user", "content": [{"type": "text", "text": "What type of bed is present in the image?\n"}, {"type": "image"}]}, {"role": "assistant", "content": [{"type": "text", "text": "The bed in the image is a yellow bed. In Minecraft, beds are used for sleeping to skip to dawn and to set a player's respawn point. Beds come in various colors, and the color is determined by the dye used in crafting. The yellow bed is crafted using yellow dye and wool.\n"}]}]}

#from mc-vqa-241102.jsonl
'''
{"id": "6940125a-15fc-4776-9dc5-ae75cf0f4949", 
"label": ["vqa", "contractor", "Minecraft"], 
"image": ["images/mc-vqa-241102/6940125a-15fc-4776-9dc5-ae75cf0f4949.jpg"], "datetime": "24_11_02", 
"model": "gpt_4o", "conversations": 
[{"role": "user", "content": [{"type": "text", "text": "What type of bed is present in the image?\n"}, {"type": "image"}]}, 
{"role": "assistant", "content": [{"type": "text", "text": "The bed in the image is a yellow bed. In Minecraft, beds are used for sleeping to skip to dawn and to set a player's respawn point. Beds come in various colors, and the color is determined by the dye used in crafting. The yellow bed is crafted using yellow dye and wool.\n"}]}]}
'''

# as we can see, we get a file path (relative) the user prompt (no system prompt) and the assistant output we're trying to predict
# btw this sample is from mc-vqa ... .jsonl , let's see if other formats differ in any other ways
# they actually vary in format which sucks as it means more code

# from mc-grounding-point-gui.jsonl
'''
{"id": "0de6ae27-3dce-4e21-83fa-10d7f85212e1", 
"label": ["gui", "Minecraft", "point"], "image": ["images/mc-grounding-point-gui/0de6ae27-3dce-4e21-83fa-10d7f85212e1.jpg"], 
"conversations": 
[{"role": "user", "content": [{"type": "text", "text": "Point directly to the exact location of the recipe book in the image.\n"}, 
{"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[79.1, 40.1]], "label": "recipe book"}]}], 
"source": {"bbox": [[415, 72], [431, 86]], "image_source": "/public/hkc/craft_cold_start/craft_inv/frames_valid/640/frame_119.jpg"}}
'''
# here the format seems to be user prompt + image tokens, and then we try to predict the points

# mc-grounding-point-embodied.jsonl
'''
{"id": "No.Grounding.6xx.Player261-f153ac423f61-20211022-162743.393.chest", 
"label": ["embody", "Minecraft", "point"], 
"conversations": 
[{"role": "user", "content": [{"type": "text", "text": "Pinpoint the chest.\n"}, {"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[82.1875, 83.88888888888889]], "label": "chest"}]}], 
"datetime": "2025-02-21 12:50:19", "model": "SAM2", 
"image": ["images/mc-grounding-point-embodied/6xx-Player261-f153ac423f61-20211022-162743-393.jpg"], 
"source": {"image_url": "images/grounding-event-sam2-0221/6xx-Player261-f153ac423f61-20211022-162743-393.jpg", 
"video_path": "/nfs-shared/data/contractors/all_6xx_Jun_29/videos/Player261-f153ac423f61-20211022-162743.mp4",
 "points": [302, 526], "bbox": [409, 236, 639, 359], "label": "chest"}}
'''
# this one is also of the format user prompt + image -> single point prediction, so similar to the one above 

#mc-grounding-point-embodied-image5.jsonl

'''
{"id": "No.Grounding.6xx.Player670-f153ac423f61-20210810-105232.798.800.evoker",
"label": ["embody", "Minecraft", "point"],
 "conversations":
 [
{"role": "user", "content": [{"type": "text", "text": "Pin down the evoker in the image, providing its location in points.\n"}, {"type": "image", "text": "<image>"}]},
{"role": "assistant", "content": [{"type": "point", "point": [[53.75, 42.77777777777778]], "label": "evoker"}]},
{"role": "user", "content": [{"type": "image", "text": "<image>"}]},
{"role": "assistant", "content": [{"type": "point", "point": [[51.87500000000001, 47.22222222222222]], "label": "evoker"}]}],
"datetime": "2025-02-21 17:36:36", "model": "SAM2", "image": ["images/mc-grounding-point-embodied-image5/6xx-Player670-f153ac423f61-20210810-105232-798.jpg", "images/mc-grounding-point-embodied-image5/6xx-Player670-f153ac423f61-20210810-105232-800.jpg"
],
"source": {"image_urls": ["images/grounding-event-sam2-image=5-0222/6xx-Player670-f153ac423f61-20210810-105232-798.jpg", "images/grounding-event-sam2-image=5-0222/6xx-Player670-f153ac423f61-20210810-105232-800.jpg"], 
"video_path": "/nfs-shared/data/contractors/all_6xx_Jun_29/videos/Player670-f153ac423f61-20210810-105232.mp4",
"points": [[154, 344], [170, 332]], "bbox": [[220, 63, 437, 239], [216, 80, 416, 255]], "label": "evoker", "action": "kill_entity:evoker"}}
'''

# again here, we also seem to have some kind of user prompt + image tokens -> point

#mc-caption-241104.jsonl
'''
{"id": "88140ef9-3c2e-4abb-abf0-585799c725e3", 
"label": ["caption", "contractor", "question_based", "Minecraft"], 
"image": ["images/mc-caption-241104/88140ef9-3c2e-4abb-abf0-585799c725e3.jpg"], "datetime": "24_11_04", "model": "gpt_4o", 
"conversations": [
{"role": "user", "content": [{"type": "text", "text": "Discuss the context and content visible in this Minecraft scene.\n"}, 
{"type": "image"}]},
{"role": "assistant", "content": [{"type": "text", "text": "This Minecraft image is taken from a first-person perspective, offering an immersive view of the game's blocky world. The scene unfolds on a hilly terrain, where natural elements like grass and dirt meet the player's modifications. A path made of wooden planks leads up the hill, suggesting previous player activity in shaping the landscape. The area where the dirt, grass, and cobblestone meet is the main focal point, possibly indicating a path or structure that has been built or altered by the player. The mix of textures, including cobblestone, dirt, grass, and wooden planks, adds depth to the scene. The landscape is a blend of natural generation and player alteration, with the wooden plank paths and cobblestone structures standing out against the naturally occurring grass and dirt.\n"}]}]}
'''

# here its descriptive work, still in user, image assistant

#overall all 5 seem to all have the user + image tokens -> assistant ground truth, and though the semantics/nature of the stuff
# we want to predict varies, it is always the same format meaning we can build a "global" function to handle that data 
# (we'll do so without caching the processed data this time)

'\n{"id": "88140ef9-3c2e-4abb-abf0-585799c725e3", \n"label": ["caption", "contractor", "question_based", "Minecraft"], \n"image": ["images/mc-caption-241104/88140ef9-3c2e-4abb-abf0-585799c725e3.jpg"], "datetime": "24_11_04", "model": "gpt_4o", \n"conversations": [\n{"role": "user", "content": [{"type": "text", "text": "Discuss the context and content visible in this Minecraft scene.\n"}, \n{"type": "image"}]},\n{"role": "assistant", "content": [{"type": "text", "text": "This Minecraft image is taken from a first-person perspective, offering an immersive view of the game\'s blocky world. The scene unfolds on a hilly terrain, where natural elements like grass and dirt meet the player\'s modifications. A path made of wooden planks leads up the hill, suggesting previous player activity in shaping the landscape. The area where the dirt, grass, and cobblestone meet is the main focal point, possibly indicating a path or structure that has been built or altered by the player. The mix of textur

In [12]:
# testing rq
import json
import re

#mc-caption-241104.jsonl format
with open("./data_phase_two.nosync/mc-caption-241104.jsonl", "r") as f:
    data = json.loads(next(f)) # we just want one line

user = data['conversations'][0]['content'][0]['text']
assistant = data['conversations'][1]['content'][0]['text']
image_path = data['image'][0]

print(f'{user}\n{assistant}\n{image_path}\n')

# this is the right query format for the conversation one
# promlem is that the grounding one has a different format in the conversation array
# the image5 mcgrounding one is more problematic 
# here since we have multiple user -> assistant -> user back and forths, the sets of logits we'll want to get won't be 
# from a single continuous sequence of tokens (they'll be disjoint)

print('image5 section --------- \n')
with open("./data_phase_two.nosync/mc-grounding-point-embodied-image5.jsonl", "r") as f:
    data = json.loads(next(f)) # we just want one line

# there can be multiple user and assistant strings
all_users = [s['content'] for s in  data['conversations'] if s['role']=='user'] # we want to extract all user prompts first
all_assistants = [s['content'] for s in  data['conversations'] if s['role']=='assistant']
all_images = data['image']
users = ""

assistants = ""
complete = ""
# only the user portion of the prompt
for user in all_users:
    for content in user:
        if content['type']== 'text' or content['type']=='image':# or content['type']=='point':
            users+=content['text'] # way too nested

for assistant in all_assistants:
    for content in assistant:
        for point in content['point']:
            assistants+='<|POINT_START|>'+str(point)+'<|POINT_END|>'
        assistants += '<|LABEL_START|>'+content['label']+'<|LABEL_END|>'



print(f' TOTAL AGENTS : \n {all_users + all_assistants}\n')
print(f' TOTAL IMAGES : \n {all_images}\n')
print(f' USER PROMPTS : {users}\n') # this hands us the sort of complete user section of the prompt
print(f' ASSISTANT PROMPTS : {assistants}\n')
print(f' TOTAL PROMPT : {complete}')


## let's try to build this in a way that doesn't de-assemble and re-asseble it now that we got the possible formats of these jsons
## It seems that the format is : singleton text prompt + for(n in number of blocks){ image prompt, assistant point/points, label} 
# let's try to walk this cleanly in a general way such that it works for all grounding-point based jsons for which we have a total of 3
# hence why we don't want to rebuild a function for all three

#mc-grounding-point-embodied-image5.jsonl sample
'''
{"id": "No.Grounding.6xx.Player670-f153ac423f61-20210810-105232.798.800.evoker",
"label": ["embody", "Minecraft", "point"],
 "conversations":
 [
{"role": "user", "content": [{"type": "text", "text": "Pin down the evoker in the image, providing its location in points.\n"}, {"type": "image", "text": "<image>"}]},
{"role": "assistant", "content": [{"type": "point", "point": [[53.75, 42.77777777777778]], "label": "evoker"}]},
{"role": "user", "content": [{"type": "image", "text": "<image>"}]},
{"role": "assistant", 
"content": [{"type": "point", "point": [[51.87500000000001, 47.22222222222222]], 
"label": "evoker"}]}
],
"datetime": "2025-02-21 17:36:36", "model": "SAM2", "image": ["images/mc-grounding-point-embodied-image5/6xx-Player670-f153ac423f61-20210810-105232-798.jpg", "images/mc-grounding-point-embodied-image5/6xx-Player670-f153ac423f61-20210810-105232-800.jpg"],
"source": {"image_urls": ["images/grounding-event-sam2-image=5-0222/6xx-Player670-f153ac423f61-20210810-105232-798.jpg", "images/grounding-event-sam2-image=5-0222/6xx-Player670-f153ac423f61-20210810-105232-800.jpg"], 
"video_path": "/nfs-shared/data/contractors/all_6xx_Jun_29/videos/Player670-f153ac423f61-20210810-105232.mp4",
"points": [[154, 344], [170, 332]], "bbox": [[220, 63, 437, 239], [216, 80, 416, 255]], "label": "evoker", "action": "kill_entity:evoker"}}
'''


# we want to have a general parser for all 3 grounding jsons, which will be kinda hard but better off than building three fucking parsers
def general_part_two_json_parser(json_filepath): # image folder e.g mc-caption-241104
    with open(json_filepath, 'r') as f:
        lines = [json.loads(s) for s in f]
        # what we want to return for each line/json object is a prompt STRING that may vary in size, points, ect...
    prompts = []
    for line in lines: #every line is a json object
        r_str = ""
        img_count = 0
        for e in line['conversations']:
            if e['role']=='user':
                for obj, i in zip(e['content'], range(len(e['content']))):
                    if i == 0 and obj['type']=='text': # the first element of our conversations array is always the singleton text prompt
                        # somehow content is an array with a single element
                        r_str+=(f'<|BEG_PROMPT|><|BEG_TEXT_PROMPT|>{obj['text']}<|END_TEXT_PROMPT|>')
                        # once we've got the text prompt we need to get the image that usually sits in the 0-index slot of the array, but for the 1st one it will sit at the 1-index one, but we could pattern match on it rather than trying to make shit up
                        # here we don't increment the user element count
                        continue #to not re-append our singleton text prompt
                    elif obj['type']=='image': # we use .get() to filter out the None scenario
                        # we don't give a fuck about the image urls here, path is already in the label
                        r_str += f'<|IMAGE_PATH_TOKEN_BEG|>./data_phase_two.nosync/{line["image"][img_count]}<|IMAGE_PATH_TOKEN_END|>'
                        img_count+= 1

            elif e['role']=='assistant':
                # now we need to add the points and labels
                first_key = e['content'][0]['type'] # we'll need that to filter the point outputs from the text ones
                if first_key == 'text':
                    r_str+=(f'<|ASSISTANT_BEG|>{e['content'][0]['text']}<|ASSISTANT_END|>')
                elif first_key == 'point':
                    for point in e['content']: # point is an array of numbers btw
                        for (x,y) in point['point']:
                            r_str+=(f'<|POINT_BEG|>{f'[{int(10*x)}, {int(10*y)}]'}<|POINT_END|>') # truncating down our number strings as they will get tokenzied
                            # btw we multiply by 10 to get a result in 0,1000 range as specified by the paper as the current results are in %
                        r_str+=(f'<|LABEL_BEG|>{point['label']}<|LABEL_END|>') # there is only a single string per assistant data object
                else:
                    raise RuntimeError('non-point non-text assistant output')
            elif e['role']=='image':
                # for the caption kind mc-caption-241104.jsonl
                r_str+=(f'<|IMAGE_PATH_TOKEN_BEG|>./data_phase_two.nosync/{line['image'][0]}<|IMAGE_PATH_TOKEN_END|>') # also stored in the

        prompts.append((r_str+'<|END_PROMPT|>')) # at the end of the for loop

    return prompts
                    
# our parser also works for the mc-grounding-point-embodied.jsonl format too which is good
# (we had to add a fork for image_url in source instead of image_urls) (holy formatting fork bomb, this dataset is hell to parse or
# maybe am I just really bad at parsing ?)
'''
{"id": "No.Grounding.6xx.Player261-f153ac423f61-20211022-162743.393.chest",,
"label": ["embody", "Minecraft", "point"], 
"conversations": 
[{"role": "user", "content": [{"type": "text", "text": "Pinpoint the chest.\n"}, {"type": "image", "text": "<image>"}]}, 
{"role": "assistant", "content": [{"type": "point", "point": [[82.1875, 83.88888888888889]], "label": "chest"}]}
], 
"datetime": "2025-02-21 12:50:19", "model": "SAM2", 
"image": ["images/mc-grounding-point-embodied/6xx-Player261-f153ac423f61-20211022-162743-393.jpg"], 
"source": {"image_url": "images/grounding-event-sam2-0221/6xx-Player261-f153ac423f61-20211022-162743-393.jpg", 
"video_path": "/nfs-shared/data/contractors/all_6xx_Jun_29/videos/Player261-f153ac423f61-20211022-162743.mp4",
 "points": [302, 526], "bbox": [409, 236, 639, 359], "label": "chest"}}
'''

# how bout the grounding kind ?

# from mc-grounding-point-gui.jsonl
'''
{"id": "c193681d-5b56-4627-aa17-e98223af87b2_5", 
"label": ["gui", "Minecraft", "point"], "image": ["images/mc-grounding-point-gui/c193681d-5b56-4627-aa17-e98223af87b2_4_2_5.jpg"], 
"conversations": 
[
{"role": "user", "content": 
[{"type": "text", "text": "Identify the position of the red dye in the image, expressing the location in points.\n"}, 
{"type": "image", "text": "<image>"}]}, {"role": "assistant", "content": [{"type": "point", "point": [[93.1, 47.8]], 
"label": "red dye"}]}
], 
"datatime": "2025-01-23 01:46:02", 
"model": "shell", 
"source": {
"image_url": ["images/bbox-2d-craft-0120/c193681d-5b56-4627-aa17-e98223af87b2_4.jpg"], 
"label": "red dye", 
"points": [{"x": 784, "y": 378}], 
"bbox": [{"label": "red dye", "bbox": [[[90.9, 44.8], [95.3, 50.8]]]}]}
}
''' 

# we need another fork to fetch the image kind again, this dataset is a fucking fork bomb, I hate it

# now let's also try to properly parse the captions

# mc-caption-241104.jsonl
# for vqa
'''
{"id": "88140ef9-3c2e-4abb-abf0-585799c725e3", 
"label": ["caption", "contractor", "question_based", "Minecraft"], 
"image": ["images/mc-caption-241104/88140ef9-3c2e-4abb-abf0-585799c725e3.jpg"], 
"datetime": "24_11_04", 
"model": "gpt_4o", 
"conversations": 
[
{"role": "user", "content": [{"type": "text", "text": "Discuss the context and content visible in this Minecraft scene.\n"},
{"type": "image"}
]},
{"role": "assistant", "content": [{"type": "text", "text": "This Minecraft image is taken from a first-person perspective, offering an immersive view of the game's blocky world. The scene unfolds on a hilly terrain, where natural elements like grass and dirt meet the player's modifications. A path made of wooden planks leads up the hill, suggesting previous player activity in shaping the landscape. The area where the dirt, grass, and cobblestone meet is the main focal point, possibly indicating a path or structure that has been built or altered by the player. The mix of textures, including cobblestone, dirt, grass, and wooden planks, adds depth to the scene. The landscape is a blend of natural generation and player alteration, with the wooden plank paths and cobblestone structures standing out against the naturally occurring grass and dirt.\n"}]}
]
}
'''

Please provide a caption for this Minecraft screenshot.

In the heart of a dimly lit cave, a tense and adventurous scene unfolds. The main focal point is a zombie, positioned centrally and directly in front of the player, its presence contributing to the overall sense of danger. The surrounding environment is sparse, with stone walls that focus attention on the immediate threat. As the zombie approaches, the area immediately around it becomes the focal point, drawing the player's attention to the impending hostile encounter.

The cave's landscape, typical of an underground system, features a visible lava pool that not only poses a hazard but also serves as a recognizable landmark for navigation. The presence of lava and stone walls suggests that this area may offer resource-gathering opportunities, such as mining for stone or possibly valuable ores like diamonds or obsidian. However, the current atmosphere is one of tension and survival, rather than exploration or resource collection.


'\n{"id": "88140ef9-3c2e-4abb-abf0-585799c725e3", \n"label": ["caption", "contractor", "question_based", "Minecraft"], \n"image": ["images/mc-caption-241104/88140ef9-3c2e-4abb-abf0-585799c725e3.jpg"], \n"datetime": "24_11_04", \n"model": "gpt_4o", \n"conversations": \n[\n{"role": "user", "content": [{"type": "text", "text": "Discuss the context and content visible in this Minecraft scene.\n"},\n{"type": "image"}\n]},\n{"role": "assistant", "content": [{"type": "text", "text": "This Minecraft image is taken from a first-person perspective, offering an immersive view of the game\'s blocky world. The scene unfolds on a hilly terrain, where natural elements like grass and dirt meet the player\'s modifications. A path made of wooden planks leads up the hill, suggesting previous player activity in shaping the landscape. The area where the dirt, grass, and cobblestone meet is the main focal point, possibly indicating a path or structure that has been built or altered by the player. The mix of

This parser is lowkey useless, let's do another one

In [13]:
import json

def general_part_two_json_parser(json_filepath, root='./data_phase_two.nosync'):
    texts, paths = [], []
    with open(json_filepath) as f:
        for l in f:
            line = json.loads(l)
            images = [f"{root}/{p}" for p in line['image']]
            k, s = 0, ""
            for turn in line['conversations']:
                content = turn['content'] if isinstance(turn['content'], list) else [{"type": "text", "text": turn['content']}]
                body, points = "", []
                for obj in content:
                    if obj['type'] == 'text':
                        body += obj['text']
                    elif obj['type'] == 'image':
                        body += "<|vision_start|><|image_pad|><|vision_end|>"  # path goes in `images`
                        k += 1
                    elif obj['type'] == 'point':  # 0-100 -> Qwen3-VL 0-1000
                        for p in obj['point']:
                            x, y = (p['x'], p['y']) if isinstance(p, dict) else p
                            points.append({"point_2d": [min(max(round(x*10), 0), 1000), min(max(round(y*10), 0), 1000)],
                                           "label": obj.get('label', '')})
                    else:
                        raise ValueError(f"unknown content type {obj['type']!r} in row {line.get('id')}")
                if points:
                    body += json.dumps(points)
                s += f"<|im_start|>{turn['role']}\n{body}<|im_end|>\n"
            if k != len(images):
                raise ValueError(f"{k} image slots but {len(images)} paths in row {line.get('id')}")
            texts.append(s)
            paths.append(images)
    return texts, paths

In [14]:
import json, os, glob

ROOT = './data_phase_two.nosync'

for path in sorted(glob.glob(f'{ROOT}/*.jsonl')):
    n = mismatch = missing = 0
    first_bad = None
    with open(path) as f:
        for i, l in enumerate(f):
            d = json.loads(l)
            n += 1
            slots = sum(1 for m in d['conversations']
                          for c in (m['content'] if isinstance(m['content'], list) else [])
                          if c.get('type') == 'image')
            imgs = d.get('image', [])
            if slots != len(imgs):
                mismatch += 1
                first_bad = first_bad or (i, 'slots', slots, len(imgs))
            for p in imgs:
                if not os.path.exists(os.path.join(ROOT, p)):
                    missing += 1
                    first_bad = first_bad or (i, 'missing', p)
    print(os.path.basename(path), n, 'mismatch:', mismatch, 'missing:', missing, first_bad)

mc-caption-241104.jsonl 9023 mismatch: 0 missing: 0 None
mc-grounding-point-embodied-image5.jsonl 2439 mismatch: 0 missing: 0 None
mc-grounding-point-embodied.jsonl 29710 mismatch: 0 missing: 0 None
mc-grounding-point-gui.jsonl 195515 mismatch: 0 missing: 0 None
mc-vqa-241102.jsonl 24774 mismatch: 0 missing: 0 None


In [15]:
# thank God we now have a fully generalized json parser for all 5 kinds of fucking files
# before training our model on any bogus, let's try it out

import os

def check(jsonl):
    texts, paths = general_part_two_json_parser(jsonl)
    ok_paths = all(os.path.exists(p) for row in paths for p in row)
    ok_slots = all(t.count('<|image_pad|>') == len(row) for t, row in zip(texts, paths))
    print(len(texts), ok_paths, ok_slots)
    return ok_paths and ok_slots

check('./data_phase_two.nosync/mc-grounding-point-embodied-image5.jsonl')
check('./data_phase_two.nosync/mc-grounding-point-embodied.jsonl')
check('./data_phase_two.nosync/mc-grounding-point-gui.jsonl')
check('./data_phase_two.nosync/mc-caption-241104.jsonl')
check('./data_phase_two.nosync/mc-vqa-241102.jsonl')


2439 True True
29710 True True
195515 True True
9023 True True
24774 True True


True

In [ ]:
# testing the VQAs assumptions we made on format 
texts, _ = general_part_two_json_parser('./data_phase_two.nosync/mc-vqa-241102.jsonl')
multi = [t for t in texts if t.count('<|im_start|>assistant') > 1]
print(len(texts), len(multi))
if multi: print(multi[0])

# we actually do have multi-turn user - agent - user back and forths

24774 23891
<|im_start|>user
How many spruce planks are being crafted in this image?
<|vision_start|><|image_pad|><|vision_end|><|im_end|>
<|im_start|>assistant
In the image, 12 spruce planks are being crafted. In Minecraft, crafting one spruce wood block yields 4 spruce planks. There are three spruce wood blocks in the crafting grid, so 3 blocks x 4 planks each = 12 spruce planks.
<|im_end|>
<|im_start|>user
How many crafting recipe slots are filled with spruce logs?
<|im_end|>
<|im_start|>assistant
In the crafting grid shown in the image, there are three slots filled with spruce logs.
<|im_end|>
<|im_start|>user
Is there a crafting table visible in this image?
<|im_end|>
<|im_start|>assistant
Yes, there is a crafting table visible in the image. The crafting interface shown on the screen indicates that the player is using a crafting table, as it displays a 3x3 crafting grid. This grid size is only available when using a crafting table, as opposed to the 2x2 grid available in the playe

In [20]:

# first of all, we need to unfreeze all the ViT/vision related parameters of our Qwen VLM
import math

for p in base_model.model.visual.parameters():
    p.requires_grad = True # this time it is on


# then we set up the optimizer, learning rate and everything, we only want to pass in the parameters that have required_grad = True ofc
# as well as the parameters specified by the paper (I think default for AdamW B2 is like 0.999 so we do need to pass em explicitly)
trainable_params = [p for p in base_model.parameters() if p.requires_grad == True]
optimizer = torch.optim.AdamW(
    params = trainable_params,
    betas=(0.9, 0.95),
    eps=1e-8,
    lr=5e-6,
    weight_decay=0
)

# let's define our max_steps, since the paper doesn't specifiy the number of steps/epochs they used, I'll assume 1 epoch
B = 8

# now our cosine lr w/ warmup function we'll use in the loop
lr_max = 5e-6
warmup_steps = 200
# warmup of 200 steps

def get_lr(step):                
    warmup = 200 * accum              
    if step <= warmup:
        return step * (lr_max / warmup)
    progress = (step - warmup) / (max_step - warmup)
    return 0.5 * lr_max * (1 + math.cos(progress * math.pi))
    

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu'
import torch.nn.functional as F

# actually rather than building the batch inside the iters, we might want to make it into a function, maybe for later use
eval_every = 150

# this time we need a pre-processor, to separate the file paths from the rest of the text and insert a path token instead 
root = './data_phase_two.nosync/'
text, path =  general_part_two_json_parser(root+'mc-caption-241104.jsonl')
text2, path2 = general_part_two_json_parser(root+'mc-grounding-point-embodied-image5.jsonl')
text3, path3 = general_part_two_json_parser(root+'mc-grounding-point-embodied.jsonl')
text4, path4 = general_part_two_json_parser(root+'mc-grounding-point-gui.jsonl')
text5, path5 = general_part_two_json_parser(root+'mc-vqa-241102.jsonl')
text = text + text2 + text3 + text4 + text5
path = path + path2 + path3 + path4 + path5
# let's put them into tuples to shuffle them while keeping order
tups = [(t,p) for t,p in zip(text,path)]
import random
random.Random(67).shuffle(tups)
# now that it is truly shuffled let's split them back
shuf_text, shuf_path = [], []
for tup in tups:
    shuf_text.append(tup[0])
    shuf_path.append(tup[1])



#print(shuf_text[0:3])
#print(shuf_path[0:3])

# let's now split our sets (1% test)
split = len(shuf_text) - int(len(shuf_text)*0.01) # 99% mark
# train
txt_train = shuf_text[:split]
path_train = shuf_path[:split]
# test
txt_test = shuf_text[split:]
path_test = shuf_path[split:]

max_step = int(len(txt_train)/B)
from PIL import Image

def build_mixed_batch_tensors(texts, paths):
    images = [Image.open(p).convert("RGB") for row in paths for p in row]
    batch = base_processor(text=texts, images=images, padding=True, return_tensors="pt")
    return batch



def build_targets(ids): # for the masked y
    target = torch.full_like(ids, -67)
    for r, line in enumerate(ids.tolist()):
        for i in range(len(line) - 2):
            if line[i] == 151644 and line[i+1] == 77091: # every <|im_start|>assistant
                j = i + 3
                end = line.index(151645, j) # its own <|im_end|>
                target[r, j:end+1] = ids[r, j:end+1]
    return target


In [21]:
for f in ['mc-vqa-241102.jsonl', 'mc-grounding-point-gui.jsonl']:
    t, p = general_part_two_json_parser(f'./data_phase_two.nosync/{f}')
    batch = build_mixed_batch_tensors(t[:1], p[:1])
    target = build_targets(batch['input_ids'])
    print(f, '\n', tokenizer.decode(batch['input_ids'][0][target[0] != -67]), '\n')

mc-vqa-241102.jsonl 
 In the image, 12 spruce planks are being crafted. In Minecraft, crafting one spruce wood block yields 4 spruce planks. There are three spruce wood blocks in the crafting grid, so 3 blocks x 4 planks each = 12 spruce planks.
<|im_end|>In the crafting grid shown in the image, there are three slots filled with spruce logs.
<|im_end|>Yes, there is a crafting table visible in the image. The crafting interface shown on the screen indicates that the player is using a crafting table, as it displays a 3x3 crafting grid. This grid size is only available when using a crafting table, as opposed to the 2x2 grid available in the player's inventory.
<|im_end|> 

mc-grounding-point-gui.jsonl 
 [{"point_2d": [438, 754], "label": "2 slot"}]<|im_end|> 



In [ ]:
# Phase II Training loop :
# we'll also want to log our loss to draw nice curves
import os, time
run_id = time.strftime("%Y%m%d-%H%M%S")
log_dir = f"phase_two_logs/{run_id}"
os.makedirs(log_dir, exist_ok=True)

train_log = open(f"{log_dir}/train.csv", "w"); train_log.write("step,loss,lr\n")
test_log  = open(f"{log_dir}/test.csv", "w");  test_log.write("step,loss\n")
# we'll use this function to compute test loss
@torch.no_grad()
def test_loss(bs=B):
    base_model.eval()
    total, n_tok = 0.0, 0
    for i in range(0, len(txt_test), bs):
        batch, _ = build_mixed_batch_tensors(txt_test[i:i+bs], path_test[i:i+bs])
        target = build_targets(batch['input_ids']).to(device)
        batch = batch.to(device)
        logits = base_model(**batch).logits
        total += F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                                 target[:, 1:].reshape(-1),
                                 ignore_index=-67, reduction="sum").item()
        n_tok += (target[:, 1:] != -67).sum().item()
    base_model.train()
    return total / n_tok

accum = 256 // B
# as said in the paper we want 256-long batches (256 sequences)
# hence that will require 256 forward passes of the llm, and multiplex into 256*lenght of the sequences (post-system+user content) loss computations
base_model.train() # flipping the value
optimizer.zero_grad()
for step in range(max_step):
    # we need to set up the batch
    # let's remember that our ds is in the form [(idx, tokens)] where ids is where the assistant part of the sequence beings
    batch_text, batch_path = txt_train[step*B:(step+1)*B], path_train[step*B:(step+1)*B] # start is inclusive, end exclusive
    batch_tok = build_mixed_batch_tensors(batch_text, batch_path) 
    # now that we have our batched tensor, let's push them through the llm of Qwen3VL
    # now we need to find the start assistant start token to build our mask such that we don't train to
    # let's first find the pivot
   
    target = build_targets(batch_tok['input_ids'])
    batch_tok = batch_tok.to(device)
    target = target.to(device)
    if step % accum == 0:
        optimizer.zero_grad() # zeroing out the gradient values first
    out = base_model(**batch_tok) # ** syntax unpacks everything
    logits = out.logits # we get something of the shape (B, len, vocab) i.e one logit distribution per token which is what we wanted
    loss = F.cross_entropy(
        logits[:,:-1].reshape(-1, logits.shape[-1]), # (B*len, vocab)
        target[:, 1:].reshape(-1),
        ignore_index=-67
    )
    (loss/accum).backward()
    if (step +1)% accum == 0:
        opt_step = step // accum
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0) # gradient clippingn
        for g in optimizer.param_groups: g["lr"] = get_lr(opt_step*accum)
        optimizer.step()

        train_log.write(f"{opt_step},{loss.item()},{get_lr(opt_step*accum)}\n"); train_log.flush()
        if opt_step % 50 == 0:
            print(f"step {opt_step} | loss {loss.item():.4f} | lr {get_lr(opt_step*accum):.2e}")

        if opt_step % eval_every == 0 or step == max_step - 1: # test loss every 200
            tl = test_loss()
            test_log.write(f"{opt_step},{tl}\n"); test_log.flush()
            print(f"  test loss {tl:.4f}")

    if step == max_step-1: # since phase one is kind of short, it is ok to only save at the end
        base_model.save_pretrained(f"checkpoints/{run_id}/phase2")


In [25]:
# sampling the size of the shards to see how much we should test on
from huggingface_hub import HfApi
tree = HfApi().list_repo_tree("CraftJarvis/minecraft-vla-sft",path_in_repo='data', repo_type="dataset")
sizes = [f.size for f in tree if f.path.endswith(".parquet")]
print(f"{sizes[0]/1e9:.2f} GB per shard, {sum(sizes)/1e9:.0f} GB total, {len(sizes)} shards")
files = HfApi().list_repo_files("CraftJarvis/minecraft-vla-sft", repo_type="dataset")
print([f for f in files if f.endswith(".parquet")][:10], len(files))

0.49 GB per shard, 106 GB total, 217 shards
['data/train-00000-of-00216.parquet', 'data/train-00001-of-00216.parquet', 'data/train-00002-of-00216.parquet', 'data/train-00003-of-00216.parquet', 'data/train-00004-of-00216.parquet', 'data/train-00005-of-00216.parquet', 'data/train-00006-of-00216.parquet', 'data/train-00007-of-00216.parquet', 'data/train-00008-of-00216.parquet', 'data/train-00009-of-00216.parquet'] 219


In [ ]:
# PHASE 3 : Action training 

# the optimal action chunk size per-the paper is 1 for Post-training, followed by size 3 for the fine-tuning stage (which isn't
#  really detailed but just seems to be phase 3 w/ chunk size 3 rather than 1)

data_step_three = snapshot_download( 
    repo_id="CraftJarvis/minecraft-vla-sft",
    repo_type="dataset",
    local_dir="./data_phase_three.nosync",
    allow_patterns=['data/train-00000-of-00216.parquet', 'data/train-00001-of-00216.parquet', 'data/train-00002-of-00216.parquet', 'data/train-00003-of-00216.parquet']
)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 591.02it/s]


In [9]:
# we need to unpack the .parquet

import pandas as pd

df = pd.read_parquet('./data_phase_three.nosync/data/train-00000-of-00216.parquet') 
# let's read one parquet rq
print(df.shape, df.columns.tolist())
print(df.iloc[0])


(17484, 5) ['id', 'label', 'image', 'conversations', 'image_bytes']
id               jumpy-denim-lion-5304af046010-20220105-224029-...
label            [trajectory, RT2, mine_block:acacia_leaves, h=...
image            [/data/hf_datasets/openai-conditioned-trajecto...
conversations    [{'role': 'user', 'content': [{'type': 'text',...
image_bytes      [b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x0...
Name: 0, dtype: object


In [10]:
# I am not sure if the 51 least frequently used tokens are already embedded into the dataset as the actions or if we need to convert
# the specific actions to their mapping in that 51 set, the ds uses :
'''
<|reserved_special_token_178|><|reserved_special_token_220|><|reserved_special_token_243|><|reserved_special_token_179|>"
'''
# so let's see if those tokenize into actual tokens or are just placehodlers for us to replace
print(tokenizer.tokenize("<|point_start|>"))


['<', '|', 'point', '_start', '|', '>']


Those are not standard tokens, we hence need to map them to the 51 LFU tokens 

In [12]:
# the paper actually lies about recycling LFU tokens, it just adds new action tokens to the vocabulary, so I guess 
# we'll also do that using our special_tokens.json array from their repo
import json
with open('./special_tokens.json') as f:
    special_tokens = json.load(f)

base_processor.tokenizer.add_special_tokens({'additional_special_tokens' : special_tokens})
# let's now try to tokenize some of our special tokens again and see if they do return 1 token 
print(tokenizer.tokenize("<|point_start|>"))
print(tokenizer.tokenize('<|reserved_special_token_248|>'))
print(tokenizer.tokenize('<|reserved_special_token_59|>'))
print(len(base_processor.tokenizer), base_model.get_input_embeddings().weight.shape[0])
# to see if the tokens fit in the table

['<|point_start|>']
['<|reserved_special_token_248|>']
['<|reserved_special_token_59|>']
151918 151936


In [14]:
# regarding compute, we really want to optimize our training loops as (especially for phase 3) we are more limited on compute money
from torch.utils.data import DataLoader
# than the paper, on the rented Lambda cloud H100, we could load all shards into the CPU ram, and run 
# To be able to fully utilize the GPU's compute, we need to optimize the dataloading from CPU to make sure it doesn't 
# bottleneck us, btw we also want to load the rows into an arrow object rather than a python one to not get copy on read
# let's first load all shard (all 4 here, but in prod we'd load everything) (this notebook will cosplay main cpu process for now)

import pyarrow.parquet as pq

table = pq.read_table(
    "./data_phase_three.nosync/data",
    columns=["conversations", "image_bytes", "label"],   # skip `id`/`image`, you don't need them
)
print(table.num_rows, table.nbytes / 1e9, "GB") # we keep labels for logging, not really needed

69936 1.988606255 GB


In [15]:
# let's first check the format per row, to see how many user-assistant back and forths we have
import json

from collections import Counter
convs = table.column("conversations").to_pylist()
print(Counter(sum(m["role"] == "assistant" for m in c) for c in convs))   # assistant turns per row
print(Counter(len(r) for r in table.column("image_bytes").to_pylist()))  # images per row
print(Counter(m["role"] for c in convs for m in c))                      # roles present (system?)

# ok so we do have user -> assistant -> image w/ a single instance of each, easier than the rest

print(json.dumps(convs[0], indent=2))

Counter({1: 69936})
Counter({1: 69936})
Counter({'user': 69936, 'assistant': 69936})
[
  {
    "role": "user",
    "content": [
      {
        "type": "text",
        "text": "Break the acacia leaves to collect them.\nobservation: \n"
      },
      {
        "type": "image",
        "text": "<image>"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "type": "text",
        "text": "<|reserved_special_token_178|><|reserved_special_token_204|><|reserved_special_token_216|><|reserved_special_token_230|><|reserved_special_token_179|>\n"
      }
    ]
  }
]


In [ ]:
# we will build the data loader in another file as multiple processes from those will be sparked
B = 8 # batch size 

from concurrent_dl import plumbing_ds, collate
ds = plumbing_ds(table)
batch, targets, tags = collate([ds[0], ds[1]], base_processor)

print(base_processor.tokenizer.decode(batch["input_ids"][0][targets[0] != -67]))
print(tags)
from functools import partial # we want partial as here, we want a single source of truth as for the processor
# btw on mac, this will spark a copy of the shard for every processs (worker), this is not the case on linux
loader = DataLoader(ds, batch_size=B, num_workers=16, shuffle=True, collate_fn=partial(collate, processor=base_processor))

<|reserved_special_token_178|><|reserved_special_token_204|><|reserved_special_token_216|><|reserved_special_token_230|><|reserved_special_token_179|>
<|im_end|>
[['trajectory', 'RT2', 'mine_block:acacia_leaves', 'h=0', 'c=1'], ['trajectory', 'RT2', 'mine_block:sugar_cane', 'h=0', 'c=1']]


In [ ]:
# as for the test dataset, one good thing is that the HF repo does include it on its own, so we don't have to stress about 
# how we build it as here, a lot of actions are near identical since the images were captured in 20FPS, which could cause poisoning
# if we have identical test and train data from the same sequences (i.e to break a diamond block, you gotta hold the pickaxe for 
# seconds which generates like 40 frames of one same action and same image, which could be later split between train and test, effectively
# testing the model on data it has already seen) praying that the paper did their job correctly and doesn't include such instances
# but since we want to match the paper's methodology we shhold keep their set 

data_step_three = snapshot_download( 
    repo_id="CraftJarvis/minecraft-vla-sft",
    repo_type="dataset",
    local_dir="./data_phase_three_test.nosync",
    allow_patterns=['*valid*']
)

In [ ]:
# now as for the training loop it is very similar to the one of phase 2, with the difference of this time having a DL at hand to 
# iterate over for batches